In [ ]:
import warnings
warnings.filterwarnings('ignore')

from matplotlib import pyplot as plt
import seaborn as sns

from scipy.sparse import hstack
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, TimeSeriesSplit

In [ ]:
# функция для записи прогнозов в файл
def write_to_submission_file(
    predicted_labels, out_file, target="target", index_label="session_id"
):
    predicted_df = pd.DataFrame(
        predicted_labels,
        index=np.arange(1, predicted_labels.shape[0] + 1),
        columns=[target],
    )
    predicted_df.to_csv(out_file, index_label=index_label)

In [ ]:
# загрузим обучающую и тестовую выборки
train_df = pd.read_csv("../../data/train_sessions.csv", index_col="session_id")
test_df = pd.read_csv("../../data/test_sessions.csv", index_col="session_id")

# приведем колонки time1, ..., time10 к временному формату
times = ["time%s" % i for i in range(1, 11)]
train_df[times] = train_df[times].apply(pd.to_datetime)
test_df[times] = test_df[times].apply(pd.to_datetime)

# отсортируем данные по времени
train_df = train_df.sort_values(by="time1")

print(train_df.shape, test_df.shape)

# посмотрим на заголовок обучающей выборки
train_df.head()

In [ ]:
# приведем колонки site1, ..., site10 к целочисленному формату и заменим пропуски нулями
sites = ["site%s" % i for i in range(1, 11)]
train_df[sites] = train_df[sites].fillna(0).astype("int")
train_df[sites].to_csv(
    '/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/train_sessions_text.txt', 
    sep=' ', 
    index=None, 
    header=None
)

test_df[sites] = test_df[sites].fillna(0).astype("int")
test_df[sites].to_csv(
    '/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/test_sessions_text.txt', 
    sep=' ', 
    index=None, 
    header=None
)

In [ ]:
train_df[sites]

### sample of count vectorizer

In [ ]:
cv = CountVectorizer()
X_sparse = cv.fit_transform(['site_1 site_17 site_2', 'site_2 site_2 site_1'])
print(X_sparse.todense())
print(cv.vocabulary_)

X_sparse

In [ ]:
X_sparse.nonzero()

In [ ]:
X_sparse.data

### fit CV on our data

In [ ]:
%%time

cv = CountVectorizer()

with open('/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/train_sessions_text.txt') as inp_train_file:
    X_train = cv.fit_transform(inp_train_file)
    
with open('/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/test_sessions_text.txt') as inp_test_file:
    X_test = cv.transform(inp_test_file)
    
print(X_train.shape, X_test.shape)

In [ ]:
y_train = train_df['target'].astype('int')

### train log reg

In [ ]:
logit = LogisticRegression(C=1, random_state=17)

In [ ]:
%%time
cv_scores = cross_val_score(logit, X_train, y_train, cv=5, scoring='roc_auc')

In [ ]:
cv_scores.mean()

In [ ]:
logit.fit(X_train, y_train)

In [ ]:
test_preds_logit1 = logit.predict_proba(X_test)[:, 1]

In [ ]:
test_preds_logit1

In [ ]:
# 0.91104 ROC AUC Public LB
write_to_submission_file(test_preds_logit1, '/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/logit_subm1.csv')

### Time features

In [ ]:
train_df.head()

In [ ]:
def add_time_features(df, X_sparse):
    hour = df['time1'].apply(lambda ts: ts.hour)
    morning = ((hour >= 7) & (hour <= 11)).astype('int')
    day = ((hour >= 12) & (hour <= 18)).astype('int')
    evening = ((hour >= 19) & (hour <= 23)).astype('int')
    night = ((hour >= 0) & (hour <= 6)).astype('int')
    X = hstack([X_sparse, morning.values.reshape(-1, 1), 
                day.values.reshape(-1, 1), evening.values.reshape(-1, 1), 
                night.values.reshape(-1, 1)])
    return X

In [ ]:
%%time
X_train_new = add_time_features(train_df.fillna(0), X_train)
X_test_new = add_time_features(test_df.fillna(0), X_test)

print(X_train_new.shape, X_test_new.shape)

In [ ]:
logit = LogisticRegression(C=1, random_state=17)
cv_scores = cross_val_score(logit, X_train_new, y_train, cv=5, scoring='roc_auc')
print(cv_scores)
print(cv_scores.mean())

In [ ]:
logit.fit(X_train_new, y_train)
test_preds_logit2 = logit.predict_proba(X_test_new)[:, 1]

# 0.93583 ROC AUC Public LB
write_to_submission_file(test_preds_logit2, '/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/logit_subm2.csv')

### TimeSeriesSplit(n_splits=10)


In [ ]:
time_split = TimeSeriesSplit(n_splits=10)
[(el[0].shape, el[1].shape) for el in time_split.split(X_train_new)]

In [ ]:
logit = LogisticRegression(C=1, random_state=17)
cv_scores = cross_val_score(logit, X_train_new, y_train, cv=time_split, scoring='roc_auc', n_jobs=-1)
print(cv_scores)
print(cv_scores.mean())

### Now we tune regularization parameter C.

In [ ]:
c_values = np.logspace(-2, 2, 10)

logit_grid_searcher = GridSearchCV(estimator=logit, param_grid={'C': c_values},
                                  scoring='roc_auc', n_jobs=-1, cv=time_split, verbose=1)

In [ ]:
%%time
logit_grid_searcher.fit(X_train_new, y_train)

In [ ]:
logit_grid_searcher.best_score_, logit_grid_searcher.best_params_

In [ ]:
logit = LogisticRegression(C=0.215, random_state=17)
cv_scores = cross_val_score(logit, X_train_new, y_train, cv=time_split, scoring='roc_auc', n_jobs=-1)
print(cv_scores)
print(cv_scores.mean())

In [ ]:
logit_test_pred3 = logit_grid_searcher.predict_proba(X_test_new)[:, 1]

# 0.94083 ROC AUC Public LB
write_to_submission_file(logit_test_pred3, '/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/logit_subm3.csv') # 0.94242

### add feature YYYYMM

In [ ]:
def add_ym_feature(df, X_sparse, scaler, need_scaler_fit=False):
    ym = df['time1'].apply(lambda ts: int(f'{ts.year}{ts.month:02}'))
    ym_reshaped = ym.values.reshape(-1, 1)
    
    if need_scaler_fit:
        scaled_ym = scaler.fit_transform(ym_reshaped)
    else:
        scaled_ym = scaler.transform(ym_reshaped)
    
    X = hstack([X_sparse, scaled_ym])
    
    return X

In [ ]:
%%time
scaler = StandardScaler()
X_train_new_ym = add_ym_feature(train_df.fillna(0), X_train_new, scaler, need_scaler_fit=True)
X_test_new_ym = add_ym_feature(test_df.fillna(0), X_test_new, scaler, need_scaler_fit=False)

print(X_train_new_ym.shape, X_test_new_ym.shape)

In [ ]:
c_values = np.logspace(-3, 1, 10)

logit_grid_searcher = GridSearchCV(estimator=logit, param_grid={'C': c_values},
                                  scoring='roc_auc', n_jobs=-1, cv=time_split, verbose=1)
logit_grid_searcher.fit(X_train_new_ym, y_train)
logit_grid_searcher.best_score_, logit_grid_searcher.best_params_

In [ ]:
logit_test_pred4 = logit_grid_searcher.predict_proba(X_test_new_ym)[:, 1]

# 0.94233 ROC AUC Public LB
write_to_submission_file(logit_test_pred4, '/opt/d.sidorko/ocr_research/study/mlcourse.ai/my_work/topic4/logit_subm4.csv')